<a href="https://colab.research.google.com/github/Nitesh-9009/TokensToTranslation/blob/main/thirdWeek_SOC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Week 3 = makemore part 4 (backprop ninja) + part 5 (WaveNet)**

is week do cheez -> pehle poore MLP+batchnorm ke through **manually backprop** karna (pytorch ke autograd se compare karke), phir **WaveNet** style hierarchical model banana.

note: is week ke baad koi formal assignment nahi tha, bas repo ke notes update karne the.

## main resources
- makemore part 4 (becoming a backprop ninja): https://youtu.be/q8SA3rM6ckI
- makemore part 5 (wavenet): https://youtu.be/t3YqWQ66dQI

## other resources
- matrix calculus (deeper backprop math): https://arxiv.org/abs/1802.01528
- wavenet paper (van den Oord et al. 2016): https://arxiv.org/abs/1609.03499
- deepmind wavenet blog (dilated conv wale famous animations): https://www.deepmind.com/blog/wavenet-a-generative-model-for-raw-audio

mentor ne bola tha -> papers padho. maine bhi DL ka zyada part original papers se hi samjha.

In [ ]:
# setup + dataset (wahi names.txt)
import urllib.request
import torch
import torch.nn.functional as F

urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/karpathy/makemore/master/names.txt', 'names.txt')
words = open('names.txt').read().splitlines()

chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)

block_size = 3


def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)


import random
random.seed(42)
random.shuffle(words)
n1, n2 = int(0.8 * len(words)), int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
print('train:', Xtr.shape, '| vocab:', vocab_size)

# **Part 4 = Becoming a backprop ninja**

idea -> `loss.backward()` ki jagah har operation ka gradient khud haath se nikalna, phir pytorch ke gradient se match karna. is se samajh aata hai autograd andar kya karta hai.

compare karne ke liye ek chhota `cmp` helper -> exact match aur maxdiff dono print karta hai.

In [ ]:
def cmp(name, dt, t):
    # dt = mera manual gradient, t.grad = pytorch ka gradient
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{name:16s} | exact: {str(ex):5s} | approx: {str(app):5s} | maxdiff: {maxdiff}')

In [ ]:
# network banate hai (thoda non-standard init taaki koi gradient galti se sahi na dikhe)
n_embd, n_hidden = 10, 64
g = torch.Generator().manual_seed(2147483647)

C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5 / 3) / (n_embd * block_size) ** 0.5
b1 = torch.randn(n_hidden, generator=g) * 0.1
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1
bngain = torch.randn((1, n_hidden), generator=g) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidden), generator=g) * 0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad = True

batch_size = 32
n = batch_size
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]

In [ ]:
# forward pass -> chhote chhote steps me toda hai taaki har intermediate ka gradient nikal sake
emb = C[Xb]
embcat = emb.view(emb.shape[0], -1)
# Linear layer 1
hprebn = embcat @ W1 + b1
# BatchNorm
bnmeani = 1 / n * hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff ** 2
bnvar = 1 / (n - 1) * bndiff2.sum(0, keepdim=True)   # Bessel's correction (n-1)
bnvar_inv = (bnvar + 1e-5) ** -0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact)
# Linear layer 2
logits = h @ W2 + b2
# cross entropy loss (haath se, taaki har step backprop ho sake)
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes            # numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum ** -1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# pytorch ka backward (compare ke liye).... saare intermediates ka grad retain karo
for p in parameters:
    p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, norm_logits,
          logit_maxes, logits, h, hpreact, bnraw, bnvar_inv, bnvar, bndiff2,
          bndiff, bnmeani, hprebn, embcat, emb]:
    t.retain_grad()
loss.backward()
print('loss:', loss.item())

In [ ]:
# ab poora manual backprop.... loss se peeche ki taraf, ek ek karke
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0 / n
dprobs = (1.0 / probs) * dlogprobs
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
dcounts = counts_sum_inv * dprobs
dcounts_sum = (-counts_sum ** -2) * dcounts_sum_inv
dcounts += torch.ones_like(counts) * dcounts_sum
dnorm_logits = counts * dcounts
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)
dlogits = dnorm_logits.clone()
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)
dhpreact = (1.0 - h ** 2) * dh                 # tanh ka derivative
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
dbnraw = bngain * dhpreact
dbnbias = dhpreact.sum(0, keepdim=True)
dbndiff = bnvar_inv * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
dbnvar = (-0.5 * (bnvar + 1e-5) ** -1.5) * dbnvar_inv
dbndiff2 = (1.0 / (n - 1)) * torch.ones_like(bndiff2) * dbnvar
dbndiff += (2 * bndiff) * dbndiff2
dhprebn = dbndiff.clone()
dbnmeani = (-dbndiff).sum(0)
dhprebn += 1.0 / n * (torch.ones_like(hprebn) * dbnmeani)
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)
demb = dembcat.view(emb.shape)
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        dC[Xb[k, j]] += demb[k, j]

In [ ]:
# ab match karo pytorch ke saath -> sab 'exact: True' aana chahiye
cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

sab `exact: True` -> matlab mera manual backprop pytorch ke autograd se bilkul match kar gaya. ye exercise se cross entropy aur batchnorm ke through gradient flow acha samajh aaya.

# **Part 5 = WaveNet**

abhi tak MLP context ke saare characters ko ek saath squash kar deta tha. WaveNet ka idea -> characters ko **hierarchically** (tree ki tarah) do-do karke combine karo. is se receptive field dhire dhire badhta hai (dilated convolution jaisa - deepmind blog wale animations me yahi dikhaya hai).

iske liye karpathy ne layers ko pytorch jaise module classes me likha (Linear, BatchNorm1d, Tanh, Embedding, FlattenConsecutive, Sequential). maine bhi wahi banaya. block_size ab 8 rakha.

In [ ]:
# ----- module classes (pytorch jaisa mini API) -----
class Linear:
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out), generator=g) / fan_in ** 0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])


class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps, self.momentum, self.training = eps, momentum, True
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):
        if self.training:
            dim = 0 if x.ndim == 2 else (0, 1)   # 3d input (wavenet) ke liye
            xmean = x.mean(dim, keepdim=True)
            xvar = x.var(dim, keepdim=True)
        else:
            xmean, xvar = self.running_mean, self.running_var
        self.out = self.gamma * (x - xmean) / torch.sqrt(xvar + self.eps) + self.beta
        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]


class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out

    def parameters(self):
        return []


class Embedding:
    def __init__(self, num_embeddings, embedding_dim):
        self.weight = torch.randn((num_embeddings, embedding_dim), generator=g)

    def __call__(self, IX):
        self.out = self.weight[IX]
        return self.out

    def parameters(self):
        return [self.weight]


class FlattenConsecutive:
    # n consecutive characters ko ek saath jod deta hai (yahi hierarchy banata hai)
    def __init__(self, n):
        self.n = n

    def __call__(self, x):
        B, T, C = x.shape
        x = x.view(B, T // self.n, C * self.n)
        if x.shape[1] == 1:
            x = x.squeeze(1)
        self.out = x
        return self.out

    def parameters(self):
        return []


class Sequential:
    def __init__(self, layers):
        self.layers = layers

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        self.out = x
        return self.out

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [ ]:
# wavenet ke liye block_size = 8 wala dataset
block_size = 8
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
print('new train shape:', Xtr.shape)

n_embd, n_hidden = 24, 128
g = torch.Generator().manual_seed(2147483647)

# hierarchical model -> har FlattenConsecutive(2) do characters ko merge karta hai
# 8 -> 4 -> 2 -> 1, teen levels
model = Sequential([
    Embedding(vocab_size, n_embd),
    FlattenConsecutive(2), Linear(n_embd * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, vocab_size),
])

with torch.no_grad():
    model.layers[-1].weight *= 0.1     # last layer ko chhota -> starting loss kam

parameters = model.parameters()
print('total params:', sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

In [ ]:
# training loop
max_steps = 20000
batch_size = 32

for i in range(max_steps):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]

    logits = model(Xb)
    loss = F.cross_entropy(logits, Yb)

    for p in parameters:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < 15000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    if i % 2000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')

In [ ]:
# eval (batchnorm ko eval mode me daalo)
for layer in model.layers:
    if isinstance(layer, BatchNorm1d):
        layer.training = False


@torch.no_grad()
def split_loss(X, Y):
    return F.cross_entropy(model(X), Y).item()


print('train loss:', round(split_loss(Xtr, Ytr), 4))
print('dev   loss:', round(split_loss(Xdev, Ydev), 4))

In [ ]:
# wavenet model se naam generate karo
g2 = torch.Generator().manual_seed(2147483647 + 10)
for _ in range(15):
    out = []
    context = [0] * block_size
    while True:
        logits = model(torch.tensor([context]))
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g2).item()
        context = context[1:] + [ix]
        if ix == 0:
            break
        out.append(itos[ix])
    print(''.join(out))

# **week 3 ka summary (mera notes)**

- **backprop ninja (part 4)** -> cross entropy + batchnorm + linear + embedding sab ke gradients haath se nikale aur pytorch se exact match karaya. autograd ka andar ka kaam clear hua.
- **wavenet (part 5)** -> characters ko tree ki tarah do-do karke combine kiya (FlattenConsecutive). context 8 tak badhaya, dev loss MLP se behtar aaya.
- module classes (Linear/BatchNorm1d/Tanh/Embedding/Sequential) khud likhne se pytorch ka design bhi samajh aaya.

yaha se aage koi formal assignment nahi tha. baaki concepts (RNN, LSTM, attention, transformers) final project me use kiye hai -> woh **project/** folder me hai (english -> german translator).

<a href="https://colab.research.google.com/github/Nitesh-9009/TokensToTranslation/blob/main/thirdWeek_SOC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **RNN, LSTM, attention, transformers**

last week hum mlp tak pahunche the (fixed context size). problem ye tha -> context badhao to params bhi badhte hai.

RNN isko sequence me solve karta hai.... ek hidden state ko aage carry karo aur har step pe update karo.

is week sab kuch scratch se (pytorch layers use karke, but cell khud likha) ->
rnn cell -> lstm cell -> self attention -> transformer (chhota gpt).

same names.txt use kar rahe hai taaki compare kar saku.

In [ ]:
# same names.txt download karte hai....
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/karpathy/makemore/master/names.txt', 'names.txt')
words = open('names.txt').read().splitlines()

chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print('vocab size:', vocab_size, '| names:', len(words))

# **dataset banate hai (padded)**

har naam ko '.' + naam + '.' banate hai.... input = pura naam except last char, target = ek step aage shift. batch banane ke liye sabko max length tak '.' (index 0) se pad kar dete hai.

In [ ]:
max_len = max(len(w) for w in words) + 1   # ek extra '.' ke liye


def encode_name(w):
    seq = [stoi['.']] + [stoi[c] for c in w] + [stoi['.']]
    x = seq[:-1]
    y = seq[1:]
    pad = max_len + 1 - len(seq)   # pad karo max_len tak
    x = x + [0] * pad
    y = y + [0] * pad
    return x[:max_len], y[:max_len]


X, Y = [], []
for w in words:
    x, y = encode_name(w)
    X.append(x)
    Y.append(y)
X = torch.tensor(X)
Y = torch.tensor(Y)
print('X:', X.shape, '| Y:', Y.shape)

n1 = int(0.9 * len(X))
Xtr, Ytr = X[:n1].to(device), Y[:n1].to(device)
Xdev, Ydev = X[n1:].to(device), Y[n1:].to(device)

# **RNN cell = carry one hidden state**

formula simple hai -> h_t = tanh(Wxh * x_t + Whh * h_prev).... bas ek hidden state jo har step update hota hai. do linear layers use kiye -> ek input ke liye ek pichle hidden ke liye.

In [ ]:
class RNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.Wxh = nn.Linear(input_size, hidden_size)   # input to hidden
        self.Whh = nn.Linear(hidden_size, hidden_size)  # hidden to hidden

    def forward(self, x, h):
        return torch.tanh(self.Wxh(x) + self.Whh(h))


class CharRNN(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.cell = RNNCell(emb_dim, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.hidden_size = hidden_size

    def forward(self, x):
        # x: (batch, seq_len)
        B, T = x.shape
        h = torch.zeros(B, self.hidden_size, device=x.device)
        e = self.emb(x)
        outs = []
        for t in range(T):                    # time steps pe loop.... yahi rnn ka dil hai
            h = self.cell(e[:, t], h)
            outs.append(self.fc(h))
        return torch.stack(outs, dim=1)       # (B, T, vocab)

In [ ]:
# ek hi train function dono ke liye (rnn / lstm).... forward -> backward -> clip -> step
def train_char_model(model, steps=3000, batch_size=64, lr=0.01):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for i in range(steps):
        ix = torch.randint(0, Xtr.shape[0], (batch_size,), device=device)
        xb, yb = Xtr[ix], Ytr[ix]
        logits = model(xb)                    # (B, T, vocab)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # gradient clip
        opt.step()
        if i % 500 == 0:
            print(f'step {i:5d} | loss {loss.item():.4f}')
    return model


rnn = CharRNN(vocab_size, emb_dim=24, hidden_size=128)
rnn = train_char_model(rnn)

In [ ]:
# ek-ek char generate karke naam banata hai (rnn aur lstm dono ke liye chalta hai)
@torch.no_grad()
def sample_rnn_like(model, n=10):
    model.eval()
    for _ in range(n):
        ix = 0
        out = []
        h = torch.zeros(1, model.hidden_size, device=device)
        c = torch.zeros(1, model.hidden_size, device=device)   # lstm ke liye
        for _ in range(max_len):
            e = model.emb(torch.tensor([[ix]], device=device))[:, 0]
            if isinstance(model.cell, LSTMCell):
                h, c = model.cell(e, (h, c))
            else:
                h = model.cell(e, h)
            logits = model.fc(h)
            probs = F.softmax(logits, dim=-1)
            ix = torch.multinomial(probs, 1).item()
            if ix == 0:
                break
            out.append(itos[ix])
        print(''.join(out))
    model.train()

# **LSTM cell = rnn with gates**

plain rnn lambi memory yaad nahi rakh pata (vanishing gradient wali problem). lstm 3 gates deta hai ->

- forget gate (f) -> pichli memory kitni bhoolni hai
- input gate (i) -> nayi info kitni daalni hai
- output gate (o) -> memory se kitna bahar dikhana hai

ek cell state c hota hai jo lambi memory ki tarah kaam karta hai. maine 4 gates ek saath compute kiye phir chunk se tod diya (clean rehta hai).

In [ ]:
class LSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        # ek hi linear me 4 gates (i, f, g, o).... baad me chunk kar denge
        self.i2h = nn.Linear(input_size, 4 * hidden_size)
        self.h2h = nn.Linear(hidden_size, 4 * hidden_size)
        self.hidden_size = hidden_size

    def forward(self, x, state):
        h, c = state
        gates = self.i2h(x) + self.h2h(h)
        i, f, g, o = gates.chunk(4, dim=1)
        i = torch.sigmoid(i)      # input gate
        f = torch.sigmoid(f)      # forget gate
        g = torch.tanh(g)         # candidate memory
        o = torch.sigmoid(o)      # output gate
        c = f * c + i * g         # nayi cell state
        h = o * torch.tanh(c)     # nayi hidden state
        return h, c


class CharLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.cell = LSTMCell(emb_dim, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.hidden_size = hidden_size

    def forward(self, x):
        B, T = x.shape
        h = torch.zeros(B, self.hidden_size, device=x.device)
        c = torch.zeros(B, self.hidden_size, device=x.device)
        e = self.emb(x)
        outs = []
        for t in range(T):
            h, c = self.cell(e[:, t], (h, c))
            outs.append(self.fc(h))
        return torch.stack(outs, dim=1)


lstm = CharLSTM(vocab_size, emb_dim=24, hidden_size=128)
lstm = train_char_model(lstm)
print('\nlstm se naam:')
sample_rnn_like(lstm, n=10)

# **self attention = query key value**

rnn sequential hai (ek time me ek step). attention ka idea alag hai -> har token baaki (pichle) tokens ko dekhe aur decide kare kis pe kitna dhyan dena hai.

- query (q) -> main kya dhundh raha hu
- key (k) -> mere paas kya hai
- value (v) -> agar match hua to kya doonga

wei = q @ k^T / sqrt(d) -> lower triangular mask (future dekhna mana) -> softmax -> wei @ v. yahi karpathy ka nanogpt wala exact setup hai.

In [ ]:
class Head(nn.Module):
    # ek self attention head (causal / masked)
    def __init__(self, n_embd, head_size, block_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5   # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # future band karo
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

# **multi head + transformer block = chhota gpt**

kai heads parallel me alag alag cheez pe dhyan dete hai, phir concat kar dete hai. uske baad ek feed-forward, aur residual + layernorm (jaise "attention is all you need" paper me). ye sab milke ek transformer block banta hai.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, n_embd, head_size, block_size, dropout=0.1):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(n_embd, head_size, block_size, dropout) for _ in range(n_heads)])
        self.proj = nn.Linear(n_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)   # sab heads concat
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    # transformer block.... attention + feedforward, dono me residual + layernorm
    def __init__(self, n_embd, n_heads, block_size, dropout=0.1):
        super().__init__()
        head_size = n_embd // n_heads
        self.sa = MultiHeadAttention(n_heads, n_embd, head_size, block_size, dropout)
        self.ff = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))    # residual connection
        x = x + self.ff(self.ln2(x))
        return x

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_heads, n_layers, block_size, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)   # position bhi seekhte hai
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_heads, block_size, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        tok = self.token_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        return self.head(x)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]   # sirf last block_size tokens
            logits = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
# transformer ke liye ek continuous stream banate hai (naam '.' se separate)
block_size = 16
data = []
for w in words:
    data += [stoi[c] for c in w] + [stoi['.']]
data = torch.tensor(data, dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]


def get_batch(split, batch_size=64):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


gpt = MiniGPT(vocab_size, n_embd=64, n_heads=4, n_layers=3, block_size=block_size).to(device)
print('gpt params:', sum(p.numel() for p in gpt.parameters()))
opt = torch.optim.AdamW(gpt.parameters(), lr=3e-3)

for step in range(3000):
    xb, yb = get_batch('train')
    logits = gpt(xb)
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 500 == 0:
        print(f'step {step:5d} | loss {loss.item():.4f}')

In [ ]:
# gpt se naam generate karo
start = torch.tensor([[stoi['.']]], device=device)
out = gpt.generate(start, max_new_tokens=200)[0].tolist()

text = ''.join(itos[i] for i in out)
names = [nm for nm in text.split('.') if nm]
print('generated names:')
for nm in names[:20]:
    print(' ', nm)

# **week 3 ka summary (mera notes)**

- rnn -> ek hidden state carry karo aur har step update.... sequence handle ho gaya, but lambi memory weak
- lstm -> gates (forget / input / output) + cell state -> lambi dependency better yaad rehti hai
- attention -> q, k, v se har token relevant tokens pe dhyan de sakta hai (parallel, sequential nahi)
- transformer -> multi head attention + feedforward + residual + layernorm -> chhota gpt ban gaya

ye sab milke final project (english -> german translator) ki base ban gayi. encoder-decoder wahi lstm aur attention wale ideas pe khada hai....